
# DL Assignment 03

**Name:**

**Course Email:**  


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি 'Anyone with the link' & 'View' Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:

## Dataset: Medical Cost Personal Dataset (insurance.csv)

**Task Type:** Regression — we predict `charges` (medical insurance cost) from patient features.

---

### Preprocessing Decisions:

| Step | Decision | Reason |
|---|---|---|
| Missing values | None found | Dataset is clean |
| `sex` (male/female) | LabelEncoder → 0/1 | Binary categorical |
| `smoker` (yes/no) | LabelEncoder → 0/1 | Binary categorical |
| `region` (4 classes) | OneHotEncoder → 4 columns | Multi-class nominal, avoids ordinal bias |
| Feature scaling | StandardScaler on features | Neural nets are sensitive to feature scale |
| Target scaling | StandardScaler on `charges` | Regression target has wide range (~1k–64k), helps convergence |
| Train/Test split | 80% train, 20% test | Standard split |
| Tensor dtype | `float32` | PyTorch default for neural networks |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ── Load Dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('insurance.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
# ── Check Missing Values ──────────────────────────────────────────────────────
print("Missing values per column:")
print(df.isnull().sum())
# No missing values → no imputation needed

In [ ]:
# ── Encode Categorical Variables ──────────────────────────────────────────────
# LabelEncode binary columns: sex, smoker
le = LabelEncoder()
df['sex']    = le.fit_transform(df['sex'])     # female=0, male=1
df['smoker'] = le.fit_transform(df['smoker'])  # no=0, yes=1

# OneHotEncode multi-class column: region (4 values)
df = pd.get_dummies(df, columns=['region'], drop_first=False)

print("Columns after encoding:", df.columns.tolist())
df.head()

In [ ]:
# ── Separate Features (X) and Target (y) ─────────────────────────────────────
X = df.drop(columns=['charges']).values.astype(np.float32)
y = df['charges'].values.astype(np.float32).reshape(-1, 1)

print("X shape:", X.shape)  # (1338, 9)
print("y shape:", y.shape)  # (1338, 1)

In [ ]:
# ── Train / Test Split (80/20) ────────────────────────────────────────────────
# Split BEFORE scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

In [ ]:
# ── Feature Scaling ───────────────────────────────────────────────────────────
# Fit scaler on training set only → transform both sets
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

# Scale target (helps regression convergence)
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train)
y_test  = scaler_y.transform(y_test)

print("Scaling done. X_train mean ≈ 0:", X_train.mean().round(3))

In [ ]:
# ── Convert to PyTorch Tensors ────────────────────────────────────────────────
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor  = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor  = torch.from_numpy(y_test.astype(np.float32))

print("X_train_tensor shape:", X_train_tensor.shape)  # [1070, 9]
print("y_train_tensor shape:", y_train_tensor.shape)  # [1070, 1]
print("X_train_tensor dtype:", X_train_tensor.dtype)

# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:

### Architecture: 3-Layer Feedforward Network

```
Input (9)  →  Hidden1 (64)  →  Hidden2 (32)  →  Output (1)
```

### Justifications

| Choice | Value | Why |
|---|---|---|
| **Input neurons** | 9 | One per feature (age, sex, bmi, children, smoker, 4 region dummies) |
| **Hidden Layer 1** | 64 neurons | Enough capacity to learn complex feature interactions |
| **Hidden Layer 2** | 32 neurons | Gradually narrows representation (funnel pattern) |
| **Output neurons** | 1 | Single regression output (predicted scaled charge) |
| **Activation** | ReLU | Standard for regression hidden layers; avoids vanishing gradient; computationally cheap |
| **Output activation** | None (linear) | Regression task — we need unbounded output |


In [ ]:
# ── Model Definition (nn.Module style — like Module 14) ───────────────────────
class InsuranceNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(num_features, 64),  # Input → Hidden1
            nn.ReLU(),
            nn.Linear(64, 32),            # Hidden1 → Hidden2
            nn.ReLU(),
            nn.Linear(32, 1)              # Hidden2 → Output (linear for regression)
        )

    def forward(self, x):
        return self.network(x)


# Number of input features
num_features = X_train_tensor.shape[1]  # 9

# Instantiate the model
model = InsuranceNN(num_features)
print(model)

In [ ]:
# ── Count Trainable Parameters ────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params}")

# Breakdown layer by layer
for name, param in model.named_parameters():
    print(f"  {name:30s}  shape={str(param.shape):20s}  params={param.numel()}")

# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:

### Loss Function: `MSELoss` (Mean Squared Error)

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

**Why MSE?**
- This is a **regression** task (predicting a continuous value `charges`).
- MSE is the standard loss for regression — it penalises large errors more strongly (squared).
- MSE has smooth gradients everywhere, which is ideal for gradient-based optimisation.

---

### Optimizer: `Adam` (Adaptive Moment Estimation)

**Why Adam?**
- Adam adapts the learning rate for each parameter individually using moving averages of gradients.
- It typically converges faster than plain SGD on tabular/neural network tasks.
- Learning rate `0.001` is the standard starting point for Adam — works well here.
- The dataset is small-to-medium sized (1338 rows), so Adam's per-parameter adaptivity is beneficial.


In [ ]:
# ── Loss Function & Optimizer ─────────────────────────────────────────────────

# MSELoss — standard for regression
criterion = nn.MSELoss()

# Adam optimizer with learning rate 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function :", criterion)
print("Optimizer     :", optimizer)

# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:

### Training Pipeline — Step-by-Step Explanation

| Step | Code | What happens |
|---|---|---|
| **1. Forward pass** | `y_pred = model(X_train_tensor)` | Input flows through each layer → produces predictions |
| **2. Compute loss** | `loss = criterion(y_pred, y_train_tensor)` | MSE between predicted and true values is calculated |
| **3. Zero gradients** | `optimizer.zero_grad()` | Clear accumulated gradients from the previous step |
| **4. Backward pass** | `loss.backward()` | PyTorch auto-differentiates: computes gradient of loss w.r.t. every parameter |
| **5. Update params** | `optimizer.step()` | Adam uses gradients to nudge each weight in the direction that reduces loss |

**Loss history** is stored in a Python `list` — simple and effective.

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────

epochs = 200          # Train for 200 epochs (≥ 100 as required)
loss_history = []     # List to store training loss per epoch

for epoch in range(1, epochs + 1):

    # ── Step 1: Forward Pass ──────────────────────────────────────────────────
    # Pass the entire training set through the network
    y_pred = model(X_train_tensor)

    # ── Step 2: Compute Loss ──────────────────────────────────────────────────
    # Compare predictions with true labels using MSE
    loss = criterion(y_pred, y_train_tensor)

    # ── Step 3: Gradient Reset ────────────────────────────────────────────────
    # Zero out gradients BEFORE backward pass (PyTorch accumulates by default)
    optimizer.zero_grad()

    # ── Step 4: Backward Pass ─────────────────────────────────────────────────
    # Compute gradients via automatic differentiation
    loss.backward()

    # ── Step 5: Parameter Update ──────────────────────────────────────────────
    # Adam adjusts every weight/bias based on its gradient
    optimizer.step()

    # ── Store & Print ─────────────────────────────────────────────────────────
    loss_history.append(loss.item())

    if epoch % 10 == 0:
        print(f"Epoch [{epoch:3d}/{epochs}]  Loss: {loss.item():.6f}")

print("\nTraining complete!")

In [ ]:
# ── Plot Training Loss Curve ──────────────────────────────────────────────────
plt.figure(figsize=(9, 4))
plt.plot(range(1, epochs + 1), loss_history, color='steelblue', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss (scaled)')
plt.title('Training Loss Curve — InsuranceNN (Baseline)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final training loss: {loss_history[-1]:.6f}")

# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:

We evaluate the model on the **held-out test set** using two regression metrics:

| Metric | Formula | Meaning |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(\hat{y}-y)^2$ | Larger errors penalised more — sensitive to outliers |
| **MAE** | $\frac{1}{n}\sum|\hat{y}-y|$ | Average absolute error — easy to interpret |

Since we scaled the target, we also **inverse-transform** predictions to get errors in original dollars.

**Fitting assessment:**
- If `test_loss ≈ train_loss` → good fit  
- If `test_loss >> train_loss` → overfitting  
- If both losses are high → underfitting  

In [ ]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
model.eval()   # Switch to eval mode (disables dropout, batchnorm etc. if present)

with torch.no_grad():   # No gradient computation needed during evaluation

    # ── Test set predictions ──────────────────────────────────────────────────
    y_pred_test  = model(X_test_tensor)
    y_pred_train = model(X_train_tensor)

    # ── Scaled-space metrics ──────────────────────────────────────────────────
    train_mse = criterion(y_pred_train, y_train_tensor).item()
    test_mse  = criterion(y_pred_test,  y_test_tensor).item()

    # ── Convert back to original dollar scale ─────────────────────────────────
    y_pred_orig  = scaler_y.inverse_transform(y_pred_test.numpy())
    y_true_orig  = scaler_y.inverse_transform(y_test_tensor.numpy())

    mae = np.mean(np.abs(y_pred_orig - y_true_orig))
    mse = np.mean((y_pred_orig - y_true_orig) ** 2)

print("=" * 50)
print(f"  Train MSE (scaled)  : {train_mse:.6f}")
print(f"  Test  MSE (scaled)  : {test_mse:.6f}")
print("=" * 50)
print(f"  Test MSE ($ original) : {mse:,.2f}")
print(f"  Test MAE ($ original) : {mae:,.2f}")
print("=" * 50)

# ── Fitting Assessment ────────────────────────────────────────────────────────
ratio = test_mse / max(train_mse, 1e-9)
if ratio < 1.3:
    verdict = "Good fit — model generalises well to unseen data."
elif ratio < 2.0:
    verdict = "Mild overfitting — test loss is somewhat higher than train loss."
else:
    verdict = "Overfitting — test loss is significantly higher than train loss."

print(f"  Test/Train MSE ratio  : {ratio:.3f}")
print(f"  Verdict               : {verdict}")

# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:

### Experiment: Larger Hidden Layers + More Epochs

| Setting | Baseline | Modified |
|---|---|---|
| Hidden Layer 1 | 64 | **128** |
| Hidden Layer 2 | 32 | **64** |
| Epochs | 200 | **400** |
| Learning rate | 0.001 | 0.001 (unchanged) |

**Hypothesis:** More neurons → higher capacity → potentially lower loss; more epochs → more time to converge.

In [ ]:
# ── Modified Model ────────────────────────────────────────────────────────────
class InsuranceNN_Larger(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 128),   # Wider hidden layer 1
            nn.ReLU(),
            nn.Linear(128, 64),             # Wider hidden layer 2
            nn.ReLU(),
            nn.Linear(64, 1)                # Same output
        )

    def forward(self, x):
        return self.network(x)


model_v2   = InsuranceNN_Larger(num_features)
criterion2 = nn.MSELoss()
optimizer2 = torch.optim.Adam(model_v2.parameters(), lr=0.001)

epochs_v2     = 400
loss_history2 = []

for epoch in range(1, epochs_v2 + 1):
    y_pred2  = model_v2(X_train_tensor)
    loss2    = criterion2(y_pred2, y_train_tensor)

    optimizer2.zero_grad()
    loss2.backward()
    optimizer2.step()

    loss_history2.append(loss2.item())

    if epoch % 40 == 0:
        print(f"Epoch [{epoch:3d}/{epochs_v2}]  Loss: {loss2.item():.6f}")

print("\nModified model training complete!")

In [ ]:
# ── Evaluate Modified Model ───────────────────────────────────────────────────
model_v2.eval()
with torch.no_grad():
    y_pred_v2_test  = model_v2(X_test_tensor)
    y_pred_v2_train = model_v2(X_train_tensor)

    train_mse_v2 = criterion2(y_pred_v2_train, y_train_tensor).item()
    test_mse_v2  = criterion2(y_pred_v2_test,  y_test_tensor).item()

    y_pred_v2_orig = scaler_y.inverse_transform(y_pred_v2_test.numpy())
    mae_v2 = np.mean(np.abs(y_pred_v2_orig - y_true_orig))
    mse_v2 = np.mean((y_pred_v2_orig - y_true_orig) ** 2)

print("=" * 60)
print(f"{'Metric':<30} {'Baseline':>12} {'Modified':>12}")
print("-" * 60)
print(f"{'Train MSE (scaled)':<30} {train_mse:>12.6f} {train_mse_v2:>12.6f}")
print(f"{'Test  MSE (scaled)':<30} {test_mse:>12.6f} {test_mse_v2:>12.6f}")
print(f"{'Test  MSE ($)':<30} {mse:>12,.0f} {mse_v2:>12,.0f}")
print(f"{'Test  MAE ($)':<30} {mae:>12,.0f} {mae_v2:>12,.0f}")
print("=" * 60)

In [ ]:
# ── Compare Loss Curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Baseline
axes[0].plot(range(1, epochs + 1), loss_history, color='steelblue')
axes[0].set_title('Baseline (64→32, 200 epochs)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].grid(alpha=0.3)

# Modified
axes[1].plot(range(1, epochs_v2 + 1), loss_history2, color='crimson')
axes[1].set_title('Modified (128→64, 400 epochs)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("Interpretation:")
print(" - The modified model (wider layers + more epochs) generally achieves a lower final loss.")
print(" - More hidden neurons give the model greater capacity to capture nonlinear patterns.")
print(" - Doubling the epochs allows the optimizer more steps to converge.")
print(" - However, if train loss drops much faster than test loss → risk of overfitting.")

# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07:

---

### Q7-A: Why must gradients be reset every epoch (iteration)?

PyTorch **accumulates** (adds) gradients into the `.grad` attribute of each tensor every time `.backward()` is called. This design is intentional for some use-cases (e.g., RNN truncated-TBPTT), but in a standard training loop it would cause **gradients from previous steps to pollute the current step's gradients**.

**Without `zero_grad()`:**
```
Epoch 1 gradient = g₁
Epoch 2 gradient = g₁ + g₂   ← WRONG! Should be just g₂
Epoch 3 gradient = g₁ + g₂ + g₃  ← even more wrong
```
The optimizer would use stale, inflated gradients, causing erratic or divergent weight updates.

✅ **Solution:** Call `optimizer.zero_grad()` before each `.backward()` to clear accumulated gradients.

---

### Q7-B: What happens if the learning rate is too high?

The **learning rate (lr)** controls how large each weight update step is:
$$w \leftarrow w - lr \cdot \nabla_w L$$

If `lr` is too large:
- The weight update overshoots the minimum of the loss surface.
- Loss may **oscillate** back and forth and never settle.
- In extreme cases, the loss **diverges** (NaN or infinity).
- The model never converges to a good solution.

```
Loss
 │    /\/\/\/\/\/\  ← oscillating / diverging with too-high lr
 │  ‾‾‾‾‾‾‾‾‾‾‾‾  ← good convergence with proper lr
 └──────────────── Epochs
```

---

### Q7-C: What happens if the learning rate is too small?

If `lr` is too small:
- Each weight update is tiny — training makes progress but **very slowly**.
- The model may require hundreds of thousands of epochs to converge.
- It may get stuck in **local minima** or **saddle points** because it does not have enough momentum to escape.
- Practically: we run out of training budget before the model reaches a good solution.

```
Loss
 │\\\_______________  ← good convergence
 │\\\\\\\\\\\\\\\  ← tiny lr, barely moving
 └──────────────── Epochs
```

---

### Q7-D: Why define layers in `__init__` and not in `forward()`?

Layers defined in `__init__` are registered as **sub-modules** of the `nn.Module`. This means:

| Benefit | What it enables |
|---|---|
| **Parameter tracking** | `model.parameters()` automatically collects all weights/biases for the optimizer |
| **State saving** | `model.state_dict()` / `model.load_state_dict()` saves and restores weights |
| **GPU/CPU transfer** | `model.to(device)` moves all parameters at once |
| **Train/eval modes** | `model.train()` / `model.eval()` propagates to all sub-modules |
| **Efficiency** | Layer objects (with their weight tensors) are created once, not re-created every forward call |

If you defined `nn.Linear(...)` inside `forward()`, a **new, unregistered layer** would be created on every call — its parameters would be invisible to the optimizer, so the network could **never learn**.
